In [1]:
import torch
import json
import io
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

In [2]:
import sys
sys.path.append('../')

In [3]:
from src.envs.agents.ppo_agent import PPOAgent

from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, PPOConvSolver, DQNConvSolver, DQNSolver
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, UnreachedPositionError

In [4]:
# agents_dir = '../output/2025-06-28/20250628-094635'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/200'

In [5]:
# agents_dir = '../output/2025-06-29/20250629-193625'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/3000'

In [6]:
# agents_dir = '../output/2025-07-08/20250708-143611'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/1000'

In [5]:
# agents_dir = '../../../output/2025-07-20/20250720-144146'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/2000'

In [6]:
agents_dir = '../../../output/2025-07-20/20250720-151109'
agent_id = 0
checkpoint_dir = f'{agents_dir}/agent{agent_id}/2000'

In [7]:
with open(f'{agents_dir}/agents_info.json') as f:
    agents_info = json.load(f)
    info = agents_info[agent_id]
    del info['type']

In [8]:
# checkpoint_dir = f'best_leader_v1'
# agent_id = 0

In [9]:
# checkpoint_dir = f'best_leader_v2'
# agent_id = 0

In [10]:
# checkpoint_dir = f'best_leader_v3'
# agent_id = 0

In [11]:
# checkpoint_dir = f'best_leader_v4.1'
# agent_id = 0

In [12]:
# checkpoint_dir = f'best_leader_v4.2'
# agent_id = 0

In [13]:
# checkpoint_dir = f'best_leader_v5'
# agent_id = 0

In [14]:
# checkpoint_dir = f'best_leader_v6'
# agent_id = 0

In [15]:
# checkpoint_dir = f'best_leader_v7_v1'
# agent_id = 0

In [8]:
# checkpoint_dir = f'best_leader_v8_v1'
# agent_id = 0

In [38]:
# checkpoint_dir = f'best_leader_v8_v2'
# agent_id = 0

In [64]:
# checkpoint_dir = f'best_leader_v8_v3'
# agent_id = 0

In [46]:
# checkpoint_dir = f'best_leader_v9'
# # agent_id = 0

In [47]:
# with open(f'{checkpoint_dir}/agents_info.json') as f:
#     agents_info = json.load(f)
#     info = agents_info[agent_id]
#     del info['type']

In [49]:
# agents_dir = '../../../output/2025-07-17/20250717-171320'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/2000'

In [50]:
# with open(f'{agents_dir}/agents_info.json') as f:
#     agents_info = json.load(f)
#     info = agents_info[agent_id]
#     del info['type']

In [51]:
# # agents_dir = '../../../output/2025-07-19/20250719-094522'
# agent_id = 0
# checkpoint_dir = f'{agents_dir}/agent{agent_id}/500'

In [52]:
# with open(f'../../../output/2025-07-17/20250717-171320/agents_info.json') as f:
#     agents_info = json.load(f)
#     info = agents_info[agent_id]
#     del info['type']

In [8]:
agent = PPOAgent(**info)

In [9]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [10]:
agent.train = False

In [11]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [12]:
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3)

(np.float64(0.875), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [13]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=1, n_max=2)

(np.float64(1.0), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [14]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=3, n_max=3)

(np.float64(0.25), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [15]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2)

(np.float64(1.0), np.float64(0.0), 1, np.float64(0.0), np.float64(0.0))

In [16]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['corner'])

(np.float64(1.0), np.float64(0.0), 4, np.float64(0.0), np.float64(0.0))

In [17]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['coridor'])

(np.float64(1.0), np.float64(0.0), 4, np.float64(0.0), np.float64(0.0))

In [18]:
weights = {}
for k,v in agent.model.state_dict().items():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

conv1.weight torch.Size([8, 12, 3, 3])
conv1.bias torch.Size([8])
bn1.weight torch.Size([8])
bn1.bias torch.Size([8])
bn1.running_mean torch.Size([8])
bn1.running_var torch.Size([8])
bn1.num_batches_tracked torch.Size([])
conv2.weight torch.Size([8, 8, 3, 3])
conv2.bias torch.Size([8])
bn2.weight torch.Size([8])
bn2.bias torch.Size([8])
bn2.running_mean torch.Size([8])
bn2.running_var torch.Size([8])
bn2.num_batches_tracked torch.Size([])
fc.weight torch.Size([16, 200])
fc.bias torch.Size([16])
actor.weight torch.Size([8, 16])
actor.bias torch.Size([8])
critic.weight torch.Size([1, 16])
critic.bias torch.Size([1])


In [19]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    '###########',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '###########',
]
env.width = len(env.map[0])
env.height = len(env.map)

In [20]:
from tests.utils import calculate_entities

In [21]:
player_pos = (3, 3)
explorers = [(5, 3), (1, 3)]
wanderers = [(3, 1, 1), (3, 5, 1), (3, 2, 0)]

entities = calculate_entities(player_pos, explorers, wanderers)
agent.set_env(env)
env._set_entities(entities)
env._set_players(entities, set_ids=True)

state = agent.observer.get_state(0)
# test_data = agent.episode_buffer.encode_states([state], return_tensors=False)
tensor_data = agent.episode_buffer.state_encoder.encode_states([state], return_tensors=True)

In [22]:
info = {
    'width': env.width,
    'height': env.height,
    'lines': env.map,
}

solver = PPOConvSolver(info, EXTENDED_KUTULU_ACTIONS, weights, size=agent.size)

In [23]:
np_output = solver._calculate_output([e.to_dict() for e in env._get_entites(0)], player_pos)

In [24]:
np_output

array([0.13194559, 0.17562333, 0.11676078, 0.16196674, 0.12049872,
       0.08716217, 0.10701806, 0.09902461])

In [25]:
model_output = agent.model(tensor_data)[0].detach().cpu().numpy()

In [26]:
model_output

array([[0.13194561, 0.17562334, 0.11676078, 0.16196676, 0.12049873,
        0.08716218, 0.10701807, 0.09902462]], dtype=float32)

In [27]:
# data2, data1 = zip(*weights.items())

# data1 = pkl.dumps(data1)
# data2 = pkl.dumps(data2)

In [28]:
data1 = []
data2 = []
for k, v in weights.items():
    if 'num_batches_tracked' in k:
        continue
    print(k, v.shape)
    data2.append(k)
    buffer = io.BytesIO()
    v = v.astype(np.float16)
    np.save(buffer, v)
    data1.append(buffer.getvalue())
    # data1.append(zlib.compress(buffer.getvalue(), level=9))

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

conv1.weight (8, 12, 3, 3)
conv1.bias (8,)
bn1.weight (8,)
bn1.bias (8,)
bn1.running_mean (8,)
bn1.running_var (8,)
conv2.weight (8, 8, 3, 3)
conv2.bias (8,)
bn2.weight (8,)
bn2.bias (8,)
bn2.running_mean (8,)
bn2.running_var (8,)
fc.weight (16, 200)
fc.bias (16,)
actor.weight (8, 16)
actor.bias (8,)
critic.weight (1, 16)
critic.bias (1,)


In [29]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [30]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", "mode = 'ppo_conv'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        line = line.replace("SIZE = 3", f"SIZE = {agent.size}")
        f.write(line)

In [31]:
!ls -lh ../src/game/template_submit.py

-rw-rw-r-- 1 kutulu kutulu 39K Jul 20 15:34 ../src/game/template_submit.py
